# 07 · 有意性ゲート付きの切片補正

06 の評価 2 で、「何例あれば安全に補正できるか」がユニットによって違うことが分かった。LOSO-B は N=20 で害 1%、
分布内は N=200 でも 49%、field / vendor はどの N でも 10% を超える。
配備時には目の前の施設がどの型か分からないので、N の閾値だけでは運用ルールにならない。

そこで b̂ 自身の有意性でゲートをかける：

$$\text{apply } \hat b \text{ if } |\hat b| > k\cdot \mathrm{SE}(\hat b),\ \text{else } b=0$$

- SE(b̂)：N 例の中だけでブートストラップする（復元抽出 → 係留点を再計算 → b̂ を解き直す、を 200 回）。
  bracket 端に張り付いた反復は除外せず数える。張り付き率が 10% を超えた抽出は「判定不能 → 補正なし」
- 同じゲートを 3 手法すべてに付ける：出力整合／比方式／ラベルあり切片（SE はラベルのブートストラップ）
- 評価プロトコルは 06 の評価 2 と同一。N 例の抽出は 06 と同じ乱数列なので、ゲートなしの数値は 06 と一致しなければならない

ロジックは `openidh_model/eval/gated_correction.py`（06 の `output_matching` を import して使う）。
06 の結果ファイルは読むだけで、上書きしない。

In [1]:
reps       = 50
seed       = 0
ns         = [5, 10, 20, 30, 50, 100, 200]
ks         = [1.0, 1.5, 2.0, 2.5]
n_boot     = 200
harm_threshold = 0.10
k_rule_n_from  = 20              # pre-registered: k must pass at every N >= this
ladder_n   = 20                  # the N-case rungs of the final ladder
in_06      = "results/output_matching.json"
out_json   = "results/gated_correction.json"
out_md     = "results/gated_correction.md"
out_fig    = "results/figures/gated_correction"

In [2]:
import os, sys, json, warnings
from pathlib import Path
_root = Path.cwd()
while not (_root / "data" / "metadata").exists():
    _root = _root.parent
os.chdir(_root); sys.path.insert(0, str(_root))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
from openidh_model.utils.paths import load_paths
from openidh_model.eval import output_matching as om
from openidh_model.eval import gated_correction as gc

for f in (out_json, out_md, out_fig + "_fire.png", out_fig + "_ece.png"):
    assert not Path(f).exists() or os.environ.get("GC_OVERWRITE") == "1", \
        f"{f} exists — refusing to overwrite an existing result (set GC_OVERWRITE=1 to rerun)"

UNITS = ["LOSO-A", "LOSO-B", "field 3T", "vendor Philips", "in-dist"]
paths = load_paths(os.environ.get("OPENIDH_PATHS", "configs/paths.local.yaml"))
runs = om.load_runs(paths, "runs")
r06 = json.loads(Path(in_06).read_text())
print(f"{len(runs)} runs; 06 results loaded from {in_06}")

27 runs; 06 results loaded from results/output_matching.json


## 0. 実行と検算

- **06 との一致**：ゲートなしの ECE 平均（ユニット × N × 手法）が 06 の評価 2 と一致すること
- **リーク検査**：下記

In [3]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    draws = gc.gated_curve(runs, ns=ns, reps=reps, seed=seed, ks=ks, n_boot=n_boot)
summ = gc.summarise_gated(draws, ks=ks)
summ["unit"] = pd.Categorical(summ["unit"], UNITS, ordered=True)
summ = summ.sort_values(["unit", "n", "method", "gate"]).reset_index(drop=True)

s06 = pd.DataFrame(r06["eval2_summary"]).set_index(["unit", "n", "method"])["ece_mean"]
mine = summ[summ.gate == "none"].astype({"unit": str}).set_index(["unit", "n", "method"])["ece_mean"]
diff = (mine - s06.reindex(mine.index)).abs()
checks = {"max_abs_diff_vs_06": float(diff.max()),
          "point_estimate_clip_warnings": len(w),
          "undeterminable_rate": summ[summ.gate == "none"].groupby("method")["undeterminable"].mean().to_dict(),
          "bootstrap_clip_rate_mean": draws.groupby("method")["clip_rate"].mean().to_dict()}
print(f"ungated vs 06: max |ΔECE| = {checks['max_abs_diff_vs_06']:.2e}  (06 JSON is written by pandas.to_json at 10 significant digits, so ~1e-11 is the floor)")
print("undeterminable rate by method:", {k: round(v, 3) for k, v in checks["undeterminable_rate"].items()})
print("mean bootstrap clip rate      :", {k: round(v, 4) for k, v in checks["bootstrap_clip_rate_mean"].items()})
assert checks["max_abs_diff_vs_06"] < 1e-9

ungated vs 06: max |ΔECE| = 4.79e-11  (06 JSON is written by pandas.to_json at 10 significant digits, so ~1e-11 is the floor)
undeterminable rate by method: {'label': 0.181, 'om': 0.0, 'ratio': 0.0}
mean bootstrap clip rate      : {'label': 0.1069, 'om': 0.0, 'ratio': 0.0}


### リーク検査（コードレビュー）

`gated_correction.gated_curve` の中で、ゲートの判定 `gate_decision(m, z[f], pic[f], y[f], pi_src, ...)` は
**N 例の添字 `f` で切り出した配列と、source 側の定数 `pi_src`（val の共変量から出した π̂。配備前から持っている値）だけ**を受け取る。
残り症例の添字 `ev` が登場するのは、判定がすべて確定した後の採点 `calib_metrics(z[ev], y[ev], ...)` だけ。
`y[f]`（N 例のラベル）を読むのはラベルあり手法（c）の推定とブートストラップだけで、出力整合と比方式の判定はラベルを一切読まない。
ブートストラップは N 例の中での復元抽出（`rng.integers(0, n, ...)`）。

加えて `tests/test_gated_correction.py::test_gate_ignores_the_remainder` が、残り症例のラベル・予測・係留値をすべて書き換えても
b̂・SE・張り付き率・判定不能・各 k の発火が 1 ビットも変わらないことを確認している。

## 1. 発火率（ゲートが開いた反復の割合）

出力整合。ユニット × N × k。比方式とラベルありの発火率は付録。

In [4]:
def wide(method, col, gates=None):
    s = summ[(summ.method == method) & (summ.gate != "none" if gates is None else summ.gate.isin(gates))]
    return s.pivot_table(index=["unit", "n"], columns="gate", values=col, observed=True)
fire = wide("om", "fire_rate")
display(fire.round(2))

gate                 1.0   1.5   2.0   2.5
unit           n                          
LOSO-A         5    0.56  0.41  0.25  0.16
               10   0.54  0.37  0.22  0.12
               20   0.63  0.48  0.36  0.22
               30   0.77  0.59  0.42  0.33
               50   0.82  0.70  0.55  0.43
               100  0.94  0.87  0.75  0.63
               200  0.99  0.99  0.93  0.87
LOSO-B         5    0.70  0.53  0.40  0.30
               10   0.86  0.73  0.59  0.40
               20   0.94  0.87  0.75  0.59
               30   0.99  0.97  0.93  0.82
               50   1.00  0.99  0.97  0.95
               100  1.00  1.00  1.00  1.00
               200  1.00  1.00  1.00  1.00
field 3T       5    0.45  0.31  0.20  0.15
               10   0.47  0.29  0.15  0.12
               20   0.52  0.35  0.17  0.09
               30   0.49  0.25  0.15  0.10
               50   0.56  0.37  0.21  0.11
               100  0.61  0.50  0.35  0.20
vendor Philips 5    0.46  0.25  0.17  0.09
               10   0.38  0.19  0.09  0.05
               20   0.34  0.13  0.08  0.03
               30   0.33  0.18  0.07  0.02
               50   0.28  0.13  0.03  0.01
               100  0.27  0.04  0.01  0.00
in-dist        5    0.53  0.37  0.26  0.17
               10   0.43  0.26  0.15  0.10
               20   0.42  0.24  0.11  0.06
               30   0.42  0.24  0.13  0.06
               50   0.49  0.30  0.17  0.06
               100  0.53  0.35  0.21  0.11
               200  0.64  0.45  0.30  0.18

In [5]:
KCOL = {"1.0": "#86b6ef", "1.5": "#3987e5", "2.0": "#256abf", "2.5": "#104281"}   # k is ordered: one-hue ramp
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})
fig, axs = plt.subplots(2, 3, figsize=(12, 7), sharey=True)
axes = axs.ravel()
for ax, u in zip(axes, UNITS):
    s = summ[(summ.unit == u) & (summ.method == "om")]
    for k in map(str, ks):
        r = s[s.gate == k]
        ax.plot(r["n"], r["fire_rate"], color=KCOL[k], lw=2, marker="o", ms=4, label=f"k = {k}")
    ax.set_xscale("log"); ax.set_xticks(ns); ax.set_xticklabels(ns, fontsize=8); ax.minorticks_off()
    ax.set_ylim(-0.03, 1.03); ax.grid(axis="y", color="#e6e5e0", lw=.6)
    ax.set_title(u, fontsize=10, loc="left"); ax.set_xlabel("N (first cases at the site)", fontsize=8)
    ax.tick_params(labelleft=True)
for ax in axs[:, 0]: ax.set_ylabel("share of draws where the gate opened")
h, l = axes[0].get_legend_handles_labels()
axes[-1].axis("off"); axes[-1].legend(h, l, frameon=False, loc="center left", fontsize=10,
                                      title="output matching, gate |b| > k·SE", title_fontsize=9)
fig.suptitle("Gate fire rate vs N (bootstrap SE inside the N cases, 200 resamples)", fontsize=11, x=0.01, ha="left")
fig.tight_layout(); Path(out_fig).parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_fig + "_fire.png", dpi=160); fig.savefig(out_fig + "_fire.svg"); plt.show()

### 事前予測との照合

事前予測は定性的な文だったので、判定基準を次のように数値化して当てる（基準はこの notebook を書いた時点で固定）。

| ユニット | 事前予測 | 判定基準 |
|---|---|---|
| LOSO-B | N=20 でほぼ常に発火、利得保持 | N=20 の発火率 ≥ 90%、かつゲートなしとの ECE 差 ≤ +0.005 |
| in-dist / vendor | ほぼ発火せず、害が消える | N ≥ 20 で発火率 ≤ 20%、かつ P(害>0.01) < 10% |
| field | 小 N では発火せず、N が増えると発火し始める | N ≤ 10 の発火率 ≤ 20%、かつ最大 N の発火率が N=5 より 20pt 以上高い |

k ごとに判定する。

In [6]:
def S(u, n, gate, col, method="om"):
    r = summ[(summ.unit == u) & (summ.n == n) & (summ.gate == str(gate)) & (summ.method == method)]
    return float(r[col].iloc[0]) if len(r) else np.nan
pr = []
for k in ks:
    fb = S("LOSO-B", 20, k, "fire_rate"); lb = S("LOSO-B", 20, k, "ece_mean") - S("LOSO-B", 20, "none", "ece_mean")
    pr.append({"k": k, "unit": "LOSO-B", "指標": f"N=20 発火 {fb:.0%}, 利得損失 {lb:+.3f}",
               "判定": "的中" if fb >= .9 and lb <= .005 else "外れ"})
    for u in ("in-dist", "vendor Philips"):
        s = summ[(summ.unit == u) & (summ.method == "om") & (summ.gate == str(k)) & (summ.n >= 20)]
        fmax, hmax = s["fire_rate"].max(), s["p_harm_tol"].max()
        pr.append({"k": k, "unit": u, "指標": f"N≥20 発火 最大 {fmax:.0%}, P(害>0.01) 最大 {hmax:.0%}",
                   "判定": "的中" if fmax <= .2 and hmax < .1 else "外れ"})
    s = summ[(summ.unit == "field 3T") & (summ.method == "om") & (summ.gate == str(k))].sort_values("n")
    small = s[s.n <= 10]["fire_rate"].max(); rise = s["fire_rate"].iloc[-1] - s["fire_rate"].iloc[0]
    pr.append({"k": k, "unit": "field 3T", "指標": f"N≤10 発火 最大 {small:.0%}, N={int(s.n.iloc[-1])} − N=5 = {rise:+.0%}",
               "判定": "的中" if small <= .2 and rise >= .2 else "外れ"})
prereg = pd.DataFrame(pr)
display(prereg)

,k,unit,指標,判定
0,1.0,LOSO-B,"N=20 発火 94%, 利得損失 +0.002",的中
1,1.0,in-dist,"N≥20 発火 最大 64%, P(害>0.01) 最大 31%",外れ
2,1.0,vendor Philips,"N≥20 発火 最大 34%, P(害>0.01) 最大 25%",外れ
3,1.0,field 3T,"N≤10 発火 最大 47%, N=100 − N=5 = +17%",外れ
4,1.5,LOSO-B,"N=20 発火 87%, 利得損失 +0.007",外れ
5,1.5,in-dist,"N≥20 発火 最大 45%, P(害>0.01) 最大 20%",外れ
6,1.5,vendor Philips,"N≥20 発火 最大 18%, P(害>0.01) 最大 13%",外れ
7,1.5,field 3T,"N≤10 発火 最大 31%, N=100 − N=5 = +19%",外れ
8,2.0,LOSO-B,"N=20 発火 75%, 利得損失 +0.020",外れ
9,2.0,in-dist,"N≥20 発火 最大 30%, P(害>0.01) 最大 11%",外れ


## 2. ゲート込みの性能（出力整合）

ユニット × N で、ゲートなしと各 k を並べる。ECE 平均 / P(害) / P(害>0.01)。
P(害) は「補正なしより少しでも悪化」、P(害>0.01) は「0.01 を超えて悪化」。ゲートが閉じた反復は補正なしと同じ ECE なので、害 0 として数える。

**読み方**：分布内でゲートがほぼ閉じたままなのは成功であって、「効いていない」ではない（補正すべきずれが無いので、動かないのが正解）。

In [7]:
def perf(method):
    s = summ[summ.method == method].copy()
    s["cell"] = s.apply(lambda r: f"{r.ece_mean:.3f} / {r.p_harm:.0%} / {r.p_harm_tol:.0%}", axis=1)
    t = s.pivot_table(index=["unit", "n"], columns="gate", values="cell", aggfunc="first", observed=True)
    t.insert(0, "raw", s[s.gate == "none"].set_index(["unit", "n"])["ece_raw"].round(3))
    return t[["raw", "none"] + [str(k) for k in ks]]
display(perf("om"))

gate                  raw               none                1.0                1.5                2.0                2.5
unit           n                                                                                                        
LOSO-A         5    0.082  0.102 / 59% / 51%  0.104 / 39% / 37%  0.104 / 32% / 31%  0.103 / 25% / 25%  0.096 / 16% / 16%
               10   0.082  0.075 / 37% / 28%  0.084 / 27% / 21%  0.088 / 23% / 17%  0.089 / 17% / 14%   0.087 / 10% / 9%
               20   0.082  0.061 / 23% / 17%  0.067 / 15% / 11%   0.070 / 13% / 9%   0.073 / 10% / 7%    0.075 / 5% / 3%
               30   0.082   0.056 / 19% / 9%   0.059 / 13% / 6%   0.067 / 13% / 6%   0.073 / 13% / 6%   0.075 / 10% / 5%
               50   0.082   0.052 / 12% / 5%    0.054 / 7% / 3%    0.058 / 7% / 3%    0.064 / 7% / 3%    0.068 / 7% / 3%
               100  0.083    0.049 / 6% / 2%    0.049 / 5% / 2%    0.052 / 5% / 2%    0.055 / 5% / 2%    0.060 / 5% / 2%
               200  0.084    0.048 / 3% / 2%    0.049 / 3% / 2%    0.049 / 3% / 2%    0.050 / 3% / 2%    0.052 / 3% / 2%
LOSO-B         5    0.162  0.115 / 24% / 21%  0.131 / 19% / 15%  0.148 / 19% / 15%  0.156 / 16% / 13%  0.162 / 15% / 12%
               10   0.162  0.086 / 11% / 10%    0.091 / 7% / 7%    0.102 / 7% / 7%    0.115 / 7% / 7%    0.133 / 5% / 5%
               20   0.162    0.065 / 1% / 1%    0.066 / 1% / 1%    0.072 / 1% / 1%    0.084 / 1% / 1%    0.100 / 1% / 1%
               30   0.162    0.062 / 0% / 0%    0.062 / 0% / 0%    0.063 / 0% / 0%    0.067 / 0% / 0%    0.077 / 0% / 0%
               50   0.161    0.059 / 0% / 0%    0.059 / 0% / 0%    0.059 / 0% / 0%    0.061 / 0% / 0%    0.063 / 0% / 0%
               100  0.163    0.055 / 0% / 0%    0.055 / 0% / 0%    0.055 / 0% / 0%    0.055 / 0% / 0%    0.055 / 0% / 0%
               200  0.160    0.056 / 0% / 0%    0.056 / 0% / 0%    0.056 / 0% / 0%    0.056 / 0% / 0%    0.056 / 0% / 0%
field 3T       5    0.088  0.118 / 69% / 53%  0.116 / 41% / 36%  0.110 / 28% / 24%  0.104 / 19% / 17%  0.101 / 15% / 14%
               10   0.089  0.103 / 59% / 43%  0.102 / 35% / 28%  0.100 / 25% / 19%  0.097 / 14% / 13%  0.096 / 11% / 11%
               20   0.090  0.094 / 52% / 27%  0.096 / 33% / 20%  0.095 / 21% / 13%   0.093 / 11% / 7%    0.092 / 7% / 5%
               30   0.091  0.091 / 45% / 24%  0.093 / 24% / 14%  0.093 / 15% / 10%    0.093 / 8% / 6%    0.093 / 7% / 5%
               50   0.092  0.088 / 40% / 21%  0.090 / 23% / 15%  0.091 / 17% / 11%   0.091 / 11% / 6%    0.092 / 7% / 4%
               100  0.104  0.102 / 45% / 29%  0.102 / 27% / 19%  0.102 / 19% / 15%   0.102 / 13% / 9%    0.102 / 7% / 5%
vendor Philips 5    0.085  0.130 / 86% / 67%  0.120 / 43% / 38%  0.110 / 25% / 23%  0.104 / 17% / 16%    0.097 / 9% / 9%
               10   0.086  0.114 / 84% / 63%  0.108 / 36% / 32%  0.100 / 18% / 17%    0.095 / 9% / 9%    0.091 / 5% / 5%
               20   0.090  0.108 / 81% / 55%  0.102 / 32% / 25%  0.097 / 12% / 11%    0.095 / 7% / 7%    0.092 / 3% / 3%
               30   0.092  0.105 / 78% / 49%  0.101 / 29% / 21%  0.098 / 16% / 13%    0.095 / 6% / 5%    0.093 / 2% / 1%
               50   0.098  0.106 / 66% / 44%  0.102 / 21% / 17%   0.100 / 11% / 9%    0.098 / 3% / 2%    0.098 / 1% / 1%
               100  0.137  0.145 / 65% / 47%  0.140 / 18% / 17%    0.138 / 3% / 3%    0.138 / 1% / 1%    0.137 / 0% / 0%
in-dist        5    0.044  0.090 / 85% / 72%  0.084 / 52% / 50%  0.075 / 37% / 36%  0.068 / 25% / 25%  0.059 / 17% / 17%
               10   0.045  0.072 / 79% / 58%  0.068 / 40% / 37%  0.062 / 26% / 25%  0.056 / 15% / 14%   0.053 / 10% / 9%
               20   0.045  0.060 / 70% / 47%  0.058 / 36% / 31%  0.055 / 21% / 20%  0.051 / 11% / 11%    0.049 / 6% / 6%
               30   0.047  0.057 / 69% / 42%  0.056 / 36% / 28%  0.053 / 21% / 18%  0.051 / 12% / 10%    0.049 / 6% / 6%
               50   0.047  0.054 / 63% / 33%  0.053 / 35% / 23%  0.052 / 23% / 17%  0.051 / 15% / 11%    0.049 / 6% / 5%
       

In [8]:
COL = {"ungated": "#2a78d6", "gated": "#eb6834", "label": "#1baf7a"}

## 3. トレードオフと k の選定

事前に固定した規則：**N ≥ 20 のすべての N・全ユニットで P(害>0.01) < 10% を満たす k のうち、
LOSO-B での「ゲート付き − ゲートなし」の ECE 差（N ≥ 20 の平均、ゲートが誤って閉じて失う利得）が最小のもの。同点なら小さい k。**

In [9]:
k_star, ktab = gc.choose_k(summ, "om", ks, n_from=k_rule_n_from, threshold=harm_threshold)
# the trade-off, spelled out: harm on the units that should not move, gain kept on those that should
rows = []
for k in ["none"] + [str(k) for k in ks]:
    r = {"gate": k}
    for u in ("in-dist", "vendor Philips", "field 3T"):
        s = summ[(summ.unit == u) & (summ.method == "om") & (summ.gate == k) & (summ.n >= k_rule_n_from)]
        r[f"{u} P(害) max"] = s["p_harm"].max(); r[f"{u} P(害>.01) max"] = s["p_harm_tol"].max()
    for u in ("LOSO-A", "LOSO-B"):
        s = summ[(summ.unit == u) & (summ.method == "om") & (summ.gate == k) & (summ.n >= k_rule_n_from)].set_index("n")
        s0 = summ[(summ.unit == u) & (summ.method == "om") & (summ.gate == "none") & (summ.n >= k_rule_n_from)].set_index("n")
        r[f"{u} 利得損失 mean"] = float((s["ece_mean"] - s0["ece_mean"]).mean())
        r[f"{u} P(害>.01) max"] = s["p_harm_tol"].max()
    rows.append(r)
trade = pd.DataFrame(rows)
display(trade.round(3)); display(ktab.round(3))
nmin = gc.n_min(summ, k_star, "om", harm_threshold) if k_star is not None else None
print(f"recommended k = {k_star}   N_min (every unit P(harm>0.01) < {harm_threshold:.0%} from here on) = {nmin}")

,gate,in-dist P(害) max,in-dist P(害>.01) max,vendor Philips P(害) max,vendor Philips P(害>.01) max,field 3T P(害) max,field 3T P(害>.01) max,LOSO-A 利得損失 mean,LOSO-A P(害>.01) max,LOSO-B 利得損失 mean,LOSO-B P(害>.01) max
0,none,0.701,0.467,0.813,0.547,0.520,0.287,0.000,0.173,0.000,0.007
1,1.0,0.359,0.305,0.320,0.247,0.333,0.200,0.003,0.107,0.000,0.007
2,1.5,0.237,0.199,0.160,0.133,0.207,0.147,0.006,0.087,0.002,0.007
3,2.0,0.157,0.112,0.073,0.073,0.133,0.093,0.010,0.073,0.005,0.007
4,2.5,0.085,0.057,0.027,0.027,0.067,0.053,0.013,0.053,0.011,0.007


,k,passes,worst_p_harm_tol,worst_unit,LOSO-B_loss
0,1.0,False,0.305,in-dist,0.000
1,1.5,False,0.199,in-dist,0.002
2,2.0,False,0.112,in-dist,0.005
3,2.5,True,0.057,in-dist,0.011


recommended k = 2.5   N_min (every unit P(harm>0.01) < 10% from here on) = 20


In [10]:
fig, axs = plt.subplots(2, 3, figsize=(12, 7), sharey=True)
axes = axs.ravel()
kk = str(k_star) if k_star is not None else "2.0"
for ax, u in zip(axes, UNITS):
    s = summ[summ.unit == u]
    r = s[(s.method == "om") & (s.gate == "none")]
    ax.plot(r["n"], r["ece_raw"], color="#8a8a86", lw=1.2, ls=(0, (1, 2)), label="uncorrected")
    ax.plot(r["n"], r["ece_mean"], color=COL["ungated"], lw=2, marker="o", ms=4, label="output matching, ungated")
    r = s[(s.method == "om") & (s.gate == kk)]
    ax.plot(r["n"], r["ece_mean"], color=COL["gated"], lw=2, marker="o", ms=4, label=f"output matching, gated k={kk}")
    r = s[(s.method == "label") & (s.gate == kk)]
    ax.plot(r["n"], r["ece_mean"], color=COL["label"], lw=2, marker="o", ms=4, label=f"labelled N, gated k={kk}")
    ax.set_xscale("log"); ax.set_xticks(ns); ax.set_xticklabels(ns, fontsize=8); ax.minorticks_off()
    ax.grid(axis="y", color="#e6e5e0", lw=.6); ax.tick_params(labelleft=True)
    ax.set_title(u, fontsize=10, loc="left"); ax.set_xlabel("N (first cases at the site)", fontsize=8)
for ax in axs[:, 0]: ax.set_ylabel("ECE on remaining cases (mean)")
h, l = axes[0].get_legend_handles_labels()
axes[-1].axis("off"); axes[-1].legend(h, l, frameon=False, loc="center left", fontsize=10)
fig.suptitle("Gated vs ungated correction: ECE vs N (same draws as notebook 06)", fontsize=11, x=0.01, ha="left")
fig.tight_layout(); fig.savefig(out_fig + "_ece.png", dpi=160); fig.savefig(out_fig + "_ece.svg"); plt.show()

## 4. vendor の係留点誤差

vendor の b̂ が +0.05 にとどまった（全例ラベルなら +0.35）原因を切り分ける。出力整合は z が同じなので、
**係留点を真の有病率に差し替えた場合の b̂（= 06 の手法 e）との差は、すべて係留点の誤差に帰せられる**。

In [11]:
e1 = pd.DataFrame(r06["eval1_per_run"])
v = e1[e1.unit == "vendor Philips"].pivot_table(index="run", columns="method", values=["b", "ece"])
vpi = e1[(e1.unit == "vendor Philips") & (e1.method == "a_raw")].set_index("run")[
    ["pi_val_obs", "pi_cov_val", "pi_cov_test", "pi_true"]]
vt = pd.concat([vpi, v["b"][["c_om_cov", "e_om_oracle"]].add_prefix("b_"),
                v["ece"][["a_raw", "c_om_cov", "e_om_oracle"]].add_prefix("ece_")], axis=1)
display(vt.round(3))
vendor = {k: float(vt[k].mean()) for k in vt.columns}
vendor["explained_by_covariates"] = (vendor["pi_cov_test"] - vendor["pi_cov_val"]) / (vendor["pi_true"] - vendor["pi_val_obs"])
print(f"vendor: π̂_cov(test) = {vendor['pi_cov_test']:.1%} vs true {vendor['pi_true']:.1%} "
      f"(val observed {vendor['pi_val_obs']:.1%}); covariates explain {vendor['explained_by_covariates']:.0%} of the val→test rise")
print(f"b̂ anchored on π̂_cov = {vendor['b_c_om_cov']:+.2f}; anchored on the truth = {vendor['b_e_om_oracle']:+.2f} "
      f"→ the whole {vendor['b_e_om_oracle'] - vendor['b_c_om_cov']:+.2f} gap is anchor error")
print(f"ECE: raw {vendor['ece_a_raw']:.3f} / OM {vendor['ece_c_om_cov']:.3f} / true anchor {vendor['ece_e_om_oracle']:.3f}")
vh = summ[(summ.unit == "vendor Philips") & (summ.method == "om") & summ.gate.isin(["none", kk])].pivot_table(
    index="n", columns="gate", values=["ece_mean", "p_harm", "p_harm_tol", "fire_rate"], observed=True)
display(Markdown(f"**vendor の害：ゲートなし vs k={kk}（N 例運用）**")); display(vh.round(3))

,pi_val_obs,pi_cov_val,pi_cov_test,pi_true,b_c_om_cov,b_e_om_oracle,ece_a_raw,ece_c_om_cov,ece_e_om_oracle
run,,,,,,,,,
vendor_s0,0.26,0.252,0.309,0.35,-0.114,0.182,0.072,0.067,0.050
vendor_s1,0.26,0.252,0.309,0.35,0.019,0.301,0.075,0.081,0.098
vendor_s2,0.26,0.252,0.309,0.35,0.242,0.580,0.107,0.127,0.111


vendor: π̂_cov(test) = 30.9% vs true 35.0% (val observed 26.0%); covariates explain 63% of the val→test rise
b̂ anchored on π̂_cov = +0.05; anchored on the truth = +0.35 → the whole +0.31 gap is anchor error
ECE: raw 0.085 / OM 0.092 / true anchor 0.086


**vendor の害：ゲートなし vs k=2.5（N 例運用）**

ece_mean        fire_rate p_harm        p_harm_tol       
gate      2.5   none       2.5    2.5   none        2.5   none
n                                                             
5       0.097  0.130     0.093  0.093  0.860      0.093  0.673
10      0.091  0.114     0.053  0.047  0.840      0.047  0.633
20      0.092  0.108     0.027  0.027  0.813      0.027  0.547
30      0.093  0.105     0.020  0.020  0.780      0.013  0.487
50      0.098  0.106     0.007  0.007  0.660      0.007  0.440
100     0.137  0.145     0.000  0.000  0.647      0.000  0.473

## 5. 最終処方箋の階段

全例の段は 06 の評価 1（test 全例の ECE）。N 例の段は N=20 の抽出で、残り症例の ECE。母集団が少し違うので、同じ段の中でだけ比べる。
EM は既報で失敗（LOSO-A 0.082 → 0.141）。他のユニットでは計算していない。

In [12]:
u1 = pd.DataFrame(r06["eval1_unit"]).set_index(["unit", "method"])["ece"]
def full_row(m): return {u: float(u1.loc[(u, m)]) for u in UNITS}
def n_row(method, gate): return {u: S(u, ladder_n, gate, "ece_mean", method) for u in UNITS}
def n_row_fire(method, gate): return {u: S(u, ladder_n, gate, "fire_rate", method) for u in UNITS}
ladder = pd.DataFrame({
    "補正なし（全例）": full_row("a_raw"),
    "EM（失敗・既報）": {u: (0.141 if u == "LOSO-A" else np.nan) for u in UNITS},
    "比方式 年齢+疑い（全例）": full_row("b_ratio"),
    "出力整合 ゲートなし（全例）": full_row("c_om_cov"),
    f"補正なし（N={ladder_n} の残り）": n_row("om", "none") and {u: S(u, ladder_n, "none", "ece_raw", "om") for u in UNITS},
    f"出力整合 ゲート k={kk}（N={ladder_n}）": n_row("om", kk),
    f"ラベル {ladder_n} 例 ゲート k={kk}": n_row("label", kk),
    "oracle＝全例ラベル切片（全例）": full_row("f_label_mle"),
}).T
fires = pd.DataFrame({f"出力整合 発火率（N={ladder_n}）": n_row_fire("om", kk),
                      f"ラベル 発火率（N={ladder_n}）": n_row_fire("label", kk)}).T
display(ladder.round(3)); display(fires.round(2))

,LOSO-A,LOSO-B,field 3T,vendor Philips,in-dist
補正なし（全例）,0.082,0.162,0.088,0.085,0.044
EM（失敗・既報）,0.141,NaN,NaN,NaN,NaN
比方式 年齢+疑い（全例）,0.070,0.064,0.090,0.092,0.044
出力整合 ゲートなし（全例）,0.038,0.042,0.070,0.092,0.044
補正なし（N=20 の残り）,0.082,0.162,0.090,0.090,0.045
出力整合 ゲート k=2.5（N=20）,0.075,0.100,0.092,0.092,0.049
ラベル 20 例 ゲート k=2.5,0.086,0.130,0.092,0.094,0.047
oracle＝全例ラベル切片（全例）,0.042,0.045,0.078,0.086,0.043


,LOSO-A,LOSO-B,field 3T,vendor Philips,in-dist
出力整合 発火率（N=20）,0.22,0.59,0.09,0.03,0.06
ラベル 発火率（N=20）,0.15,0.38,0.03,0.03,0.01


## 6. ポータルの運用ルール（擬似コード）

In [13]:
pseudo = f'''# 施設ごとに、その施設に来た症例 N 例（ラベル不要）が溜まるたびに再評価する
def site_offset(cases, k={kk}, N_min={nmin}, n_boot={n_boot}):
    N = len(cases)
    if N < N_min:
        return 0.0, "補正なし（初期値）：まだ {{N}}/{{N_min}} 例。確率値の目盛りは未補正。順位と S の利用を推奨"
    z   = [log(c.alpha / c.beta) for c in cases]          # 補正前の融合ロジット
    pi  = [pi_age_susp(c.age, c.suspicion) for c in cases]  # 疑い不明なら年齢のみの π
    b, ok = solve_mean_sigmoid(z, target=mean(pi))          # brentq, bracket [-10, 10]
    boot = [solve_mean_sigmoid(resample(z, pi)) for _ in range(n_boot)]   # N 例の中だけで復元抽出
    if not ok or clipped_share(boot) > 0.10:
        return 0.0, "補正なし：ゲート判定不能（推定が不安定）"
    se = sd(boot)
    if abs(b) <= k * se:
        return 0.0, f"補正なし：ずれの証拠なし（b̂ = {{b:+.2f}}, SE = {{se:.2f}}）"
    return b, f"補正あり：オフセット {{b:+.2f}}（SE {{se:.2f}}, N = {{N}}）"
# 表示確率 = sigmoid(z_i + offset)。S = α + β は変えない。AUC・順位は不変'''
print(pseudo)

# 施設ごとに、その施設に来た症例 N 例（ラベル不要）が溜まるたびに再評価する
def site_offset(cases, k=2.5, N_min=20, n_boot=200):
    N = len(cases)
    if N < N_min:
        return 0.0, "補正なし（初期値）：まだ {N}/{N_min} 例。確率値の目盛りは未補正。順位と S の利用を推奨"
    z   = [log(c.alpha / c.beta) for c in cases]          # 補正前の融合ロジット
    pi  = [pi_age_susp(c.age, c.suspicion) for c in cases]  # 疑い不明なら年齢のみの π
    b, ok = solve_mean_sigmoid(z, target=mean(pi))          # brentq, bracket [-10, 10]
    boot = [solve_mean_sigmoid(resample(z, pi)) for _ in range(n_boot)]   # N 例の中だけで復元抽出
    if not ok or clipped_share(boot) > 0.10:
        return 0.0, "補正なし：ゲート判定不能（推定が不安定）"
    se = sd(boot)
    if abs(b) <= k * se:
        return 0.0, f"補正なし：ずれの証拠なし（b̂ = {b:+.2f}, SE = {se:.2f}）"
    return b, f"補正あり：オフセット {b:+.2f}（SE {se:.2f}, N = {N}）"
# 表示確率 = sigmoid(z_i + offset)。S = α + β は変えない。AUC・順位は不変


## 7. 保存

In [14]:
def recs(df): return json.loads(df.reset_index().to_json(orient="records", force_ascii=False))
res = {"params": {"reps": reps, "seed": seed, "ns": ns, "ks": ks, "n_boot": n_boot,
                  "harm_threshold": harm_threshold, "k_rule_n_from": k_rule_n_from, "ladder_n": ladder_n},
       "checks": checks, "k_star": k_star, "n_min": nmin,
       "summary": recs(summ.astype({"unit": str})), "k_table": recs(ktab), "tradeoff": recs(trade),
       "prereg": recs(prereg), "vendor": vendor, "ladder": recs(ladder), "ladder_fire": recs(fires),
       "pseudocode": pseudo}
Path(out_json).write_text(json.dumps(res, indent=2, ensure_ascii=False, default=float))

def f3(x): return "  —  " if pd.isna(x) else f"{x:.3f}"
L = ["# 有意性ゲート付き切片補正 — 結果要約\n"]
L.append(f"検算：ゲートなしは 06 の評価 2 と一致（最大差 {checks['max_abs_diff_vs_06']:.0e}）。"
         f"判定不能率 出力整合 {checks['undeterminable_rate']['om']:.1%}・ラベルあり {checks['undeterminable_rate']['label']:.1%}"
         f"（ラベルありは単一クラス抽出が主）。")
L.append("リーク検査：ゲート判定は N 例の予測値・共変量（ラベルあり手法のみ N 例のラベル）と source 側定数 π̂(val) だけを読み、"
         "残り症例は判定確定後の採点にしか現れないことをコードで確認。残り症例を書き換えても判定が変わらないことをテストでも確認済み。\n")
L.append(f"## 推奨：k = {k_star}、N_min = {nmin}\n")
L.append(f"選定規則（事前固定）：N ≥ {k_rule_n_from} の全 N・全ユニットで P(害>0.01) < {harm_threshold:.0%} を満たす k のうち、LOSO-B の利得損失が最小。\n```")
L.append(f"{'k':>5}{'通過':>6}{'最悪 P(害>.01)':>16}{'最悪ユニット':>16}{'LOSO-B 損失':>12}")
for _, r in ktab.iterrows():
    L.append(f"{r['k']:>5}{'○' if r['passes'] else '×':>6}{r['worst_p_harm_tol']:>15.0%}{r['worst_unit']:>17}{r['LOSO-B_loss']:>+12.3f}")
L.append("```\n")
L.append("## 1. 発火率（出力整合、ゲートが開いた割合）\n```")
L.append(f"{'':<16}{'N':>5}" + "".join(f"{'k='+str(k):>8}" for k in ks))
for u in UNITS:
    for n in ns:
        vals = [S(u, n, k, "fire_rate") for k in ks]
        if all(np.isnan(vals)): continue
        L.append(f"{u[:14]:<16}{n:>5}" + "".join(f"{x:>8.0%}" for x in vals))
L.append("```")
L.append("事前予測：")
for _, r in prereg[prereg.k == (k_star if k_star is not None else 2.0)].iterrows():
    L.append(f"- {r['unit']}：{r['指標']} → **{r['判定']}**")
L.append("")
L.append(f"## 2. 性能（出力整合）ECE / P(害) / P(害>0.01)：ゲートなし vs k={kk}\n```")
L.append(f"{'':<16}{'N':>5}{'raw':>7}{'ゲートなし':>22}{'k='+kk:>22}")
for u in UNITS:
    for n in ns:
        a = summ[(summ.unit == u) & (summ.n == n) & (summ.method == "om")]
        if a.empty: continue
        g0, g1 = a[a.gate == "none"].iloc[0], a[a.gate == kk].iloc[0]
        c = lambda g: f"{g.ece_mean:.3f}/{g.p_harm:>4.0%}/{g.p_harm_tol:>4.0%}"
        L.append(f"{u[:14]:<16}{n:>5}{g0.ece_raw:>7.3f}{c(g0):>22}{c(g1):>22}")
L.append("```")
L.append("分布内でゲートがほぼ閉じたままなのは成功（補正すべきずれが無いので動かないのが正解）であって、「効いていない」ではない。\n")
L.append("## 3. トレードオフ（N ≥ 20 での最悪値・平均）\n```")
L.append(f"{'gate':>6}{'in-dist 害>.01':>15}{'vendor 害>.01':>15}{'field 害>.01':>14}{'LOSO-A 損失':>13}{'LOSO-B 損失':>13}")
for _, r in trade.iterrows():
    L.append(f"{r['gate']:>6}{r['in-dist P(害>.01) max']:>15.0%}{r['vendor Philips P(害>.01) max']:>15.0%}"
             f"{r['field 3T P(害>.01) max']:>14.0%}{r['LOSO-A 利得損失 mean']:>+13.3f}{r['LOSO-B 利得損失 mean']:>+13.3f}")
L.append("```\n")
L.append("## 4. vendor の係留点誤差（限界事項用）")
L.append(f"π̂_cov(vendor test) = {vendor['pi_cov_test']:.1%}、真値 {vendor['pi_true']:.1%}（val の観測は {vendor['pi_val_obs']:.1%}）。"
         f"val → test の有病率上昇のうち、共変量で説明できるのは {vendor['explained_by_covariates']:.0%}。"
         f"係留点を真値に差し替えると b̂ は {vendor['b_c_om_cov']:+.2f} → {vendor['b_e_om_oracle']:+.2f} となり、"
         f"必要量との差 {vendor['b_e_om_oracle'] - vendor['b_c_om_cov']:+.2f} は全て係留点の誤差。"
         f"ただし真値に係留しても ECE は {vendor['ece_e_om_oracle']:.3f}（補正なし {vendor['ece_a_raw']:.3f}）で、"
         f"vendor の較正誤差はそもそも切片では減らない。")
v0, v1 = vh.loc[ladder_n, ("p_harm", "none")], vh.loc[ladder_n, ("p_harm", kk)]
v0t, v1t = vh.loc[ladder_n, ("p_harm_tol", "none")], vh.loc[ladder_n, ("p_harm_tol", kk)]
L.append(f"ゲート後（N={ladder_n}）：vendor の P(害) {v0:.0%} → {v1:.0%}、P(害>0.01) {v0t:.0%} → {v1t:.0%}"
         f"（発火率 {vh.loc[ladder_n, ('fire_rate', kk)]:.0%}）。\n")
L.append(f"## 5. 最終処方箋の階段（ECE、推奨 k={kk}）\n```")
L.append(f"{'':<30}" + "".join(f"{u[:10]:>11}" for u in UNITS))
for k_, r in ladder.iterrows():
    L.append(f"{k_:<30}" + "".join(f"{f3(r[u]):>11}" for u in UNITS))
for k_, r in fires.iterrows():
    L.append(f"{k_:<30}" + "".join(f"{r[u]:>11.0%}" for u in UNITS))
L.append("```")
L.append("全例の段は test 全例、N 例の段は N=20 抽出の残り症例での ECE（同じ段の中でだけ比較）。"
         "LOSO-A / LOSO-B で出力整合（全例）が全例ラベル切片をわずかに下回るのは「oracle 超え」ではない。"
         "全例ラベル切片は NLL 最適であって ECE 最適ではなく、この差は 10 ビン ECE の揺らぎの範囲。「ラベル 0 で下限に到達」と読む。\n")
L.append("## 6. 運用ルール（擬似コード）\n```python")
L.append(pseudo)
L.append("```")
Path(out_md).write_text("\n".join(L) + "\n")
print("\n".join(L))

# 有意性ゲート付き切片補正 — 結果要約

検算：ゲートなしは 06 の評価 2 と一致（最大差 5e-11）。判定不能率 出力整合 0.0%・ラベルあり 18.1%（ラベルありは単一クラス抽出が主）。
リーク検査：ゲート判定は N 例の予測値・共変量（ラベルあり手法のみ N 例のラベル）と source 側定数 π̂(val) だけを読み、残り症例は判定確定後の採点にしか現れないことをコードで確認。残り症例を書き換えても判定が変わらないことをテストでも確認済み。

## 推奨：k = 2.5、N_min = 20

選定規則（事前固定）：N ≥ 20 の全 N・全ユニットで P(害>0.01) < 10% を満たす k のうち、LOSO-B の利得損失が最小。
```
    k    通過     最悪 P(害>.01)          最悪ユニット   LOSO-B 損失
  1.0     ×            31%          in-dist      +0.000
  1.5     ×            20%          in-dist      +0.002
  2.0     ×            11%          in-dist      +0.005
  2.5     ○             6%          in-dist      +0.011
```

## 1. 発火率（出力整合、ゲートが開いた割合）
```
                    N   k=1.0   k=1.5   k=2.0   k=2.5
LOSO-A              5     56%     41%     25%     16%
LOSO-A             10     54%     37%     22%     12%
LOSO-A             20     63%     48%     36%     22%
LOSO-A             30     77%     59%     42%     33%
LOSO-A             50     82%     70%     55%     43%
LOSO-A            10

## 付録：比方式とラベルありのゲート

In [15]:
for m in ("ratio", "label"):
    display(Markdown(f"**{m}：発火率**")); display(wide(m, "fire_rate").round(2))
    display(Markdown(f"**{m}：ECE / P(害) / P(害>0.01)**")); display(perf(m))

**ratio：発火率**

gate                 1.0   1.5   2.0   2.5
unit           n                          
LOSO-A         5    0.43  0.32  0.25  0.21
               10   0.33  0.25  0.13  0.07
               20   0.30  0.14  0.05  0.03
               30   0.28  0.13  0.03  0.01
               50   0.33  0.17  0.09  0.01
               100  0.36  0.20  0.05  0.02
               200  0.41  0.18  0.11  0.03
LOSO-B         5    0.58  0.43  0.25  0.13
               10   0.71  0.55  0.38  0.22
               20   0.90  0.79  0.63  0.47
               30   0.97  0.93  0.82  0.68
               50   1.00  0.97  0.94  0.87
               100  1.00  1.00  1.00  0.99
               200  1.00  1.00  1.00  1.00
field 3T       5    0.29  0.19  0.14  0.11
               10   0.43  0.21  0.10  0.05
               20   0.52  0.30  0.17  0.09
               30   0.63  0.39  0.16  0.09
               50   0.66  0.37  0.19  0.05
               100  0.91  0.74  0.47  0.26
vendor Philips 5    0.41  0.21  0.11  0.07
               10   0.35  0.19  0.08  0.01
               20   0.44  0.22  0.09  0.03
               30   0.48  0.23  0.11  0.02
               50   0.57  0.35  0.11  0.03
               100  0.92  0.60  0.25  0.07
in-dist        5    0.39  0.31  0.25  0.23
               10   0.29  0.15  0.10  0.07
               20   0.27  0.11  0.05  0.03
               30   0.29  0.12  0.05  0.02
               50   0.28  0.11  0.03  0.01
               100  0.28  0.12  0.03  0.00
               200  0.31  0.10  0.02  0.00

**ratio：ECE / P(害) / P(害>0.01)**

gate                  raw               none                1.0                1.5                2.0                2.5
unit           n                                                                                                        
LOSO-A         5    0.082  0.104 / 55% / 47%  0.111 / 36% / 35%  0.109 / 29% / 29%  0.106 / 24% / 24%  0.104 / 21% / 21%
               10   0.082  0.088 / 51% / 43%  0.088 / 19% / 17%  0.090 / 17% / 15%  0.090 / 12% / 11%    0.087 / 7% / 7%
               20   0.082  0.077 / 39% / 29%  0.080 / 11% / 11%    0.080 / 3% / 3%    0.082 / 2% / 2%    0.082 / 1% / 1%
               30   0.082  0.074 / 41% / 23%    0.076 / 7% / 6%    0.077 / 1% / 1%    0.081 / 1% / 0%    0.081 / 1% / 0%
               50   0.082  0.073 / 35% / 14%    0.074 / 4% / 4%    0.078 / 2% / 2%    0.079 / 1% / 1%    0.081 / 0% / 0%
               100  0.083  0.076 / 33% / 15%    0.078 / 8% / 8%    0.079 / 1% / 1%    0.082 / 1% / 1%    0.082 / 0% / 0%
               200  0.084   0.076 / 12% / 0%    0.078 / 0% / 0%    0.081 / 0% / 0%    0.082 / 0% / 0%    0.084 / 0% / 0%
LOSO-B         5    0.162  0.114 / 21% / 21%  0.137 / 17% / 17%  0.152 / 16% / 16%  0.166 / 14% / 14%  0.173 / 11% / 11%
               10   0.162    0.087 / 8% / 5%    0.099 / 4% / 3%    0.110 / 3% / 2%    0.127 / 3% / 2%    0.144 / 3% / 2%
               20   0.162    0.076 / 2% / 1%    0.078 / 1% / 1%    0.085 / 1% / 1%    0.098 / 1% / 1%    0.113 / 1% / 1%
               30   0.162    0.072 / 0% / 0%    0.072 / 0% / 0%    0.075 / 0% / 0%    0.082 / 0% / 0%    0.094 / 0% / 0%
               50   0.161    0.070 / 0% / 0%    0.070 / 0% / 0%    0.071 / 0% / 0%    0.073 / 0% / 0%    0.077 / 0% / 0%
               100  0.163    0.069 / 0% / 0%    0.069 / 0% / 0%    0.069 / 0% / 0%    0.069 / 0% / 0%    0.070 / 0% / 0%
               200  0.160    0.066 / 0% / 0%    0.066 / 0% / 0%    0.066 / 0% / 0%    0.066 / 0% / 0%    0.066 / 0% / 0%
field 3T       5    0.088  0.118 / 73% / 57%  0.111 / 28% / 27%  0.107 / 18% / 17%  0.103 / 13% / 13%  0.102 / 11% / 11%
               10   0.089  0.110 / 74% / 54%  0.107 / 37% / 35%  0.101 / 20% / 20%    0.095 / 9% / 9%    0.093 / 5% / 5%
               20   0.090  0.108 / 77% / 52%  0.108 / 47% / 40%  0.104 / 29% / 26%  0.100 / 17% / 15%    0.097 / 9% / 8%
               30   0.091  0.107 / 76% / 58%  0.106 / 52% / 45%  0.103 / 35% / 32%  0.097 / 15% / 13%    0.095 / 9% / 8%
               50   0.092  0.099 / 71% / 41%  0.098 / 47% / 33%  0.098 / 30% / 25%  0.095 / 14% / 12%    0.093 / 3% / 3%
               100  0.104  0.111 / 62% / 49%  0.111 / 58% / 47%  0.111 / 49% / 41%  0.110 / 35% / 29%  0.109 / 21% / 19%
vendor Philips 5    0.085  0.125 / 82% / 65%  0.118 / 39% / 36%  0.109 / 21% / 21%  0.101 / 11% / 11%    0.097 / 7% / 7%
               10   0.086  0.105 / 78% / 52%  0.098 / 30% / 25%  0.096 / 18% / 15%    0.091 / 8% / 7%    0.088 / 1% / 1%
               20   0.090  0.100 / 67% / 39%  0.097 / 30% / 21%  0.095 / 15% / 14%    0.092 / 7% / 6%    0.091 / 3% / 2%
               30   0.092  0.099 / 67% / 39%  0.097 / 35% / 23%  0.096 / 19% / 15%    0.094 / 9% / 8%    0.093 / 2% / 1%
               50   0.098  0.105 / 71% / 39%  0.104 / 45% / 25%  0.101 / 28% / 15%    0.099 / 9% / 5%    0.098 / 3% / 1%
               100  0.137  0.145 / 61% / 47%  0.144 / 57% / 43%  0.142 / 39% / 31%  0.140 / 17% / 14%    0.139 / 5% / 5%
in-dist        5    0.044  0.085 / 84% / 65%  0.076 / 39% / 38%  0.072 / 31% / 31%  0.066 / 25% / 25%  0.065 / 23% / 22%
               10   0.045  0.065 / 77% / 55%  0.059 / 27% / 25%  0.055 / 15% / 15%  0.052 / 10% / 10%    0.050 / 7% / 7%
               20   0.045  0.056 / 69% / 41%  0.053 / 24% / 21%  0.050 / 10% / 10%    0.048 / 5% / 5%    0.047 / 3% / 3%
               30   0.047  0.055 / 66% / 37%  0.052 / 23% / 19%  0.050 / 11% / 10%    0.049 / 5% / 4%    0.048 / 2% / 2%
               50   0.047  0.051 / 59% / 25%  0.050 / 22% / 14%    0.049 / 9% / 6%    0.048 / 3% / 3%    0.048 / 1% / 1%
       

**label：発火率**

gate                 1.0   1.5   2.0   2.5
unit           n                          
LOSO-A         5    0.03  0.01  0.00  0.00
               10   0.17  0.08  0.05  0.03
               20   0.38  0.32  0.22  0.15
               30   0.57  0.43  0.31  0.18
               50   0.79  0.64  0.50  0.33
               100  0.87  0.77  0.71  0.53
               200  0.99  0.95  0.91  0.82
LOSO-B         5    0.10  0.01  0.00  0.00
               10   0.40  0.27  0.21  0.15
               20   0.71  0.59  0.50  0.38
               30   0.89  0.81  0.69  0.59
               50   1.00  0.97  0.94  0.86
               100  1.00  1.00  0.98  0.98
               200  1.00  1.00  1.00  1.00
field 3T       5    0.01  0.01  0.00  0.00
               10   0.13  0.07  0.02  0.01
               20   0.26  0.15  0.07  0.03
               30   0.37  0.19  0.11  0.05
               50   0.47  0.30  0.17  0.10
               100  0.57  0.40  0.25  0.16
vendor Philips 5    0.01  0.00  0.00  0.00
               10   0.19  0.09  0.04  0.02
               20   0.31  0.21  0.09  0.03
               30   0.43  0.24  0.10  0.05
               50   0.46  0.29  0.15  0.03
               100  0.64  0.41  0.15  0.03
in-dist        5    0.01  0.00  0.00  0.00
               10   0.05  0.02  0.01  0.01
               20   0.11  0.04  0.03  0.01
               30   0.21  0.09  0.04  0.01
               50   0.35  0.16  0.07  0.03
               100  0.43  0.22  0.09  0.03
               200  0.50  0.32  0.15  0.05

**label：ECE / P(害) / P(害>0.01)**

gate                  raw               none                1.0                1.5                2.0                2.5
unit           n                                                                                                        
LOSO-A         5    0.082  0.121 / 49% / 44%    0.089 / 3% / 3%    0.084 / 1% / 1%    0.082 / 0% / 0%    0.082 / 0% / 0%
               10   0.082  0.087 / 44% / 33%  0.097 / 16% / 16%    0.090 / 7% / 7%    0.087 / 5% / 5%    0.086 / 3% / 3%
               20   0.081  0.075 / 41% / 33%  0.085 / 21% / 18%  0.086 / 19% / 17%  0.087 / 15% / 13%  0.086 / 11% / 10%
               30   0.082  0.061 / 21% / 15%  0.071 / 14% / 11%  0.076 / 14% / 11%  0.079 / 13% / 10%    0.082 / 8% / 7%
               50   0.082  0.059 / 21% / 14%  0.062 / 15% / 10%  0.067 / 15% / 10%  0.073 / 15% / 10%   0.077 / 11% / 7%
               100  0.083   0.052 / 12% / 8%   0.054 / 11% / 7%   0.058 / 11% / 7%   0.060 / 11% / 7%   0.067 / 11% / 7%
               200  0.084   0.052 / 10% / 5%   0.052 / 10% / 5%   0.054 / 10% / 5%   0.055 / 10% / 5%   0.057 / 10% / 5%
LOSO-B         5    0.161  0.130 / 30% / 27%    0.170 / 9% / 9%    0.164 / 1% / 1%    0.162 / 0% / 0%    0.162 / 0% / 0%
               10   0.161  0.106 / 19% / 17%   0.147 / 11% / 9%    0.156 / 9% / 8%    0.158 / 8% / 7%    0.162 / 7% / 6%
               20   0.162    0.078 / 8% / 6%    0.095 / 4% / 3%    0.109 / 4% / 3%    0.118 / 4% / 3%    0.130 / 4% / 3%
               30   0.162    0.069 / 1% / 1%    0.075 / 1% / 1%    0.082 / 1% / 1%    0.094 / 1% / 1%    0.105 / 1% / 1%
               50   0.161    0.063 / 1% / 1%    0.063 / 1% / 1%    0.065 / 1% / 1%    0.067 / 1% / 1%    0.075 / 1% / 1%
               100  0.163    0.058 / 0% / 0%    0.058 / 0% / 0%    0.058 / 0% / 0%    0.059 / 0% / 0%    0.059 / 0% / 0%
               200  0.160    0.057 / 0% / 0%    0.057 / 0% / 0%    0.057 / 0% / 0%    0.057 / 0% / 0%    0.057 / 0% / 0%
field 3T       5    0.088  0.132 / 72% / 53%    0.092 / 1% / 1%    0.090 / 1% / 1%    0.088 / 0% / 0%    0.088 / 0% / 0%
               10   0.089  0.112 / 64% / 52%  0.101 / 12% / 12%    0.097 / 7% / 7%    0.091 / 2% / 2%    0.090 / 1% / 1%
               20   0.090  0.108 / 61% / 47%  0.098 / 19% / 16%  0.097 / 13% / 11%    0.095 / 7% / 7%    0.092 / 3% / 3%
               30   0.091  0.096 / 55% / 36%  0.096 / 25% / 19%  0.095 / 14% / 11%    0.094 / 9% / 7%    0.092 / 4% / 3%
               50   0.092  0.095 / 51% / 33%  0.096 / 27% / 21%  0.095 / 16% / 14%  0.095 / 12% / 11%    0.094 / 8% / 7%
               100  0.104  0.103 / 50% / 28%  0.104 / 30% / 21%  0.104 / 22% / 16%  0.104 / 13% / 11%   0.105 / 10% / 8%
vendor Philips 5    0.085  0.147 / 93% / 75%    0.087 / 1% / 1%    0.085 / 0% / 0%    0.085 / 0% / 0%    0.085 / 0% / 0%
               10   0.086  0.129 / 84% / 67%  0.103 / 19% / 18%    0.095 / 9% / 9%    0.092 / 4% / 4%    0.090 / 2% / 2%
               20   0.090  0.115 / 83% / 60%  0.105 / 30% / 27%  0.102 / 21% / 19%    0.097 / 9% / 8%    0.094 / 3% / 3%
               30   0.092  0.107 / 71% / 45%  0.104 / 35% / 29%  0.101 / 22% / 20%   0.098 / 10% / 9%    0.096 / 5% / 5%
               50   0.098  0.108 / 70% / 44%  0.106 / 36% / 27%  0.105 / 26% / 21%  0.103 / 13% / 11%    0.099 / 3% / 3%
               100  0.137  0.143 / 56% / 40%  0.144 / 42% / 33%  0.142 / 27% / 20%  0.142 / 13% / 10%    0.139 / 3% / 3%
in-dist        5    0.044  0.113 / 85% / 72%    0.047 / 1% / 1%    0.044 / 0% / 0%    0.044 / 0% / 0%    0.044 / 0% / 0%
               10   0.045  0.082 / 84% / 64%    0.053 / 5% / 5%    0.049 / 2% / 2%    0.047 / 1% / 1%    0.046 / 1% / 1%
               20   0.045  0.067 / 75% / 53%  0.053 / 11% / 11%    0.049 / 4% / 4%    0.048 / 3% / 3%    0.047 / 1% / 1%
               30   0.047  0.063 / 74% / 50%  0.054 / 19% / 17%    0.051 / 8% / 7%    0.049 / 4% / 3%    0.047 / 1% / 1%
               50   0.047  0.058 / 68% / 45%  0.055 / 29% / 23%  0.052 / 15% / 13%    0.050 / 7% / 6%    0.049 / 2% / 2%
       